# MLOps with SageMaker
We'll build a fully automated MLOps pipeline using Amazon SageMaker Pipelines. SageMaker Pipelines lets you define, orchestrate, and manage machine learning workflows as code, making it easy to reproduce, share, and automate your ML processes.

We'll create a pipeline that:
- Processes raw data (if needed)
- Trains a model with hyperparameter tuning
- Evaluates the model against a baseline
- Registers the best model in SageMaker Model Registry

## 1. Imports and Environment Setup

To begin, we need to equip our notebook with the right tools. We need to import the Amazon SageMaker Python SDK, which acts as our remote control for AWS services.

We also establish our **Session** and **Execution Role**. Think of the Session as our open connection to SageMaker, and the Role as our ID badge that grants us permission to spin up servers and access storage. Finally, we define a default **S3 Bucket**, a cloud storage folder where all our data, training scripts, and finished models will live.

In [1]:
import sagemaker
import boto3
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import (
    ProcessingStep,
    TrainingStep,
    TuningStep,
)
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.functions import Join, JsonGet
from sagemaker.workflow.parameters import ParameterInteger, ParameterFloat, ParameterString
from sagemaker.workflow.properties import PropertyFile
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.workflow.model_step import ModelStep
from sagemaker.model import Model
from sagemaker.image_uris import retrieve

# Initialize session
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
prefix = "sagemaker/mlops-pipeline"
region = boto3.Session().region_name

print(f"Setup complete. Bucket: {bucket}, Region: {region}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Setup complete. Bucket: sagemaker-ap-southeast-2-406682760260, Region: ap-southeast-2


## 2. Simulating Data Ingestion

In a real-world enterprise scenario, data usually arrives in S3 automatically from a database or application. For this project, we are simulating that process by downloading the standard "Diabetes" dataset using Scikit-Learn.

The code below loads the data into a Pandas DataFrame, saves it locally as a CSV file, and then immediately uploads it to our S3 bucket. This uploaded file will serve as the "Raw Data" trigger that kicks off our entire machine learning pipeline.

In [2]:
import pandas as pd
from sklearn.datasets import load_diabetes

print("1. Downloading raw diabetes data...")
diabetes = load_diabetes()

# Combine features and the target into one dataframe
df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target 

# Save it locally
df.to_csv('diabetes.csv', index=False)

print("2. Uploading to S3...")
# Upload to the exact path the pipeline is expecting
s3_uri = sagemaker_session.upload_data(
    path='diabetes.csv', 
    bucket=bucket, 
    key_prefix=f'{prefix}/data/raw'
)

print(f"✅ Success! Raw data uploaded to: {s3_uri}")

1. Downloading raw diabetes data...
2. Uploading to S3...
✅ Success! Raw data uploaded to: s3://sagemaker-ap-southeast-2-406682760260/sagemaker/mlops-pipeline/data/raw/diabetes.csv


## 3. Defining Pipeline Parameters

Hardcoding values (like file paths or instance types) makes a pipeline brittle and hard to reuse. Instead, we define **Parameters**. These act as variables that can be overridden at runtime without rewriting the code.

For this pipeline, we are configuring three critical parameters:

* **`ProcessingInstanceCount` (Default: 1):**
    This controls the number of separate compute instances (servers) launched to run your data processing scripts. For small datasets like ours, `1` is sufficient. However, for massive datasets (terabytes), you could increase this to `10` or `50` to distribute the workload across a fleet of computers using frameworks like Apache Spark.

* **`TrainingInstanceType` (Default: ml.m5.large):**
    This defines the specific hardware configuration (CPU, RAM, GPU) used to train the model. By parameterizing this, we can easily balance cost vs. speed. We use standard CPU instances (`ml.m5.large`) for development to save money, but in production, we could override this parameter to use powerful GPU instances (like `ml.p3.2xlarge`) to train models significantly faster.

* **`ModelApprovalStatus` (Default: PendingManualApproval):**
    This sets the initial status of the model when it reaches the SageMaker Model Registry. It acts as a safety gate for deployment.
    * **PendingManualApproval:** The model is registered but locked. A human must review the metrics and manually "Approve" it before deployment.
    * **Approved:** The model is unlocked and ready for automatic deployment to production.

In [3]:
# S3 locations
input_data_url = ParameterString(
    name="InputDataUrl",
    default_value=f"s3://{bucket}/{prefix}/data/raw/diabetes.csv"
)

processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
training_instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.large")
model_approval_status = ParameterString(name="ModelApprovalStatus", default_value="PendingManualApproval")

## 4. creating the Preprocessing Logic (`preprocessing.py`)

Data rarely arrives ready for machine learning. It usually needs cleaning, normalization, or splitting.

The code below writes a standalone Python script named `preprocessing.py`. This script is designed to run inside a SageMaker container, completely separate from this notebook. It reads the raw input data and splits it into three distinct sets:
* **Training Set (70%):** For teaching the model.
* **Validation Set (15%):** For tuning hyperparameters.
* **Test Set (15%):** A holdout set for the final evaluation.

In [4]:
%%writefile preprocessing.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import argparse
import os

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--input-data", type=str)
    parser.add_argument("--output-train", type=str)
    parser.add_argument("--output-validation", type=str)
    parser.add_argument("--output-test", type=str)
    args = parser.parse_args()

    df = pd.read_csv(args.input_data)
    # last column is target
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

    train = pd.concat([y_train, X_train], axis=1)
    val = pd.concat([y_val, X_val], axis=1)
    test = pd.concat([y_test, X_test], axis=1)

    train.to_csv(os.path.join(args.output_train, "train.csv"), index=False, header=False)
    val.to_csv(os.path.join(args.output_validation, "validation.csv"), index=False, header=False)
    test.to_csv(os.path.join(args.output_test, "test.csv"), index=False, header=False)

Overwriting preprocessing.py


## 5. Configuring the Processing Step

Now we instruct SageMaker on how to run the script we just created. We define a **ProcessingStep** that spins up a dedicated `ml.t3.medium` instance running Scikit-Learn.

This step acts as the bridge between storage and compute: it downloads the raw data from S3, runs our `preprocessing.py` script, and then automatically uploads the resulting Train, Validation, and Test files back to S3 so the next steps in the pipeline can use them.

In [5]:
sklearn_processor = SKLearnProcessor(
    framework_version="1.0-1",
    role=role,
    instance_type="ml.t3.medium",
    instance_count=processing_instance_count,
    sagemaker_session=sagemaker_session
)

processing_step = ProcessingStep(
    name="SplitData",
    processor=sklearn_processor,
    inputs=[
        ProcessingInput(source=input_data_url, destination="/opt/ml/processing/input")
    ],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/output/train", destination=f"s3://{bucket}/{prefix}/data/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/output/validation", destination=f"s3://{bucket}/{prefix}/data/validation"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/output/test", destination=f"s3://{bucket}/{prefix}/data/test"),
    ],
    code="preprocessing.py",
    job_arguments=[
        "--input-data", "/opt/ml/processing/input/diabetes.csv",
        "--output-train", "/opt/ml/processing/output/train",
        "--output-validation", "/opt/ml/processing/output/validation",
        "--output-test", "/opt/ml/processing/output/test"
    ]
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


## 6. Automated Model Tuning

Rather than training a single model with guessed settings, we set up a **Hyperparameter Tuning Job**. This is essentially a tournament where SageMaker trains multiple versions of the model to find the best one.

#### 1. The Estimator (The Infrastructure)
First, we define the `Estimator`, which represents the actual training job infrastructure.
* **`image_uri`**: The specific Docker container image that holds the XGBoost algorithm code. We retrieve this dynamically based on our AWS region.
* **`role`**: The IAM Identity (Execution Role) that grants this training job permission to read our data from S3 and write the model back.
* **`instance_count` & `instance_type`**: The hardware we are renting. We use `1` instance of `ml.m5.large`, which is a cost-effective, general-purpose server suitable for this dataset size.
* **`output_path`**: The specific folder in our S3 bucket where the final model artifacts (the "trained brains") will be saved.

#### 2. Fixed Hyperparameters (The Baseline Rules)
These are the settings we are **not** tuning. We hold these constant across all training jobs in the tournament.
* **`objective` ("reg:squarederror")**: Tells XGBoost we are solving a **Regression** problem (predicting a continuous number) and to measure error using Squared Error.
* **`num_round` (50)**: The number of boosting rounds (decision trees) to build. We fix this at 50 to keep training fast for this demo.
* **`verbosity` (1)**: Controls how much information is logged to the console (1 = standard info).

#### 3. Hyperparameter Ranges (The Search Space)
These are the "knobs" we want SageMaker to tweak automatically. Instead of setting a single value, we provide a range.
* **`max_depth` (3-10)**: Controls how deep each decision tree can grow. Deeper trees learn more complex patterns but risk overfitting (memorizing noise).
* **`eta` (0.05-0.5)**: The "Learning Rate." This controls how quickly the model adapts. A lower `eta` makes the model more robust but slower to train; a higher `eta` learns faster but might miss the optimal solution.
* **`subsample` (0.5-1.0)**: The fraction of the training data sampled for each tree. Using less than 100% (e.g., 0.5) adds randomness that prevents the model from becoming too fixated on specific data points.

#### 4. The Tuner (The Tournament Rules)
This configures the optimization strategy.
* **`objective_metric_name` ("validation:rmse")**: The scoreboard. We tell SageMaker to judge the models based on the **Root Mean Squared Error (RMSE)** calculated on the *validation* dataset.
* **`objective_type` ("Minimize")**: Tells SageMaker that a *lower* score is better (since we want to minimize error).
* **`max_jobs` (6)**: The total number of training jobs to run in this tournament. We keep it low (6) to save cost, but in production, this might be 50+.
* **`max_parallel_jobs` (2)**: How many jobs to run at the exact same time. Running 2 at once speeds up the process.

#### 5. The Tuning Step (Pipeline Integration)
Finally, we wrap everything into a Pipeline Step.
* **`inputs`**: Connects the data. We map the `train` and `validation` output paths from the previous **ProcessingStep** to the input channels that the XGBoost algorithm expects.

In [6]:
# Get XGBoost container
container = retrieve("xgboost", region, version="1.5-1")

# Define estimator
xgb_estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type=training_instance_type,
    output_path=f"s3://{bucket}/{prefix}/models",
    sagemaker_session=sagemaker_session
)

# Set hyperparameters (basic)
xgb_estimator.set_hyperparameters(
    objective="reg:squarederror",
    num_round=50,
    verbosity=1
)

# Hyperparameter ranges
hyperparameter_ranges = {
    "max_depth": IntegerParameter(3, 10),
    "eta": ContinuousParameter(0.05, 0.5),
    "subsample": ContinuousParameter(0.5, 1.0)
}

# Create tuner
tuner = HyperparameterTuner(
    estimator=xgb_estimator,
    objective_metric_name="validation:rmse",
    objective_type="Minimize",
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=6,
    max_parallel_jobs=2
)

# Tuning step
tuning_step = TuningStep(
    name="HyperparameterTuning",
    tuner=tuner,
    inputs={
        "train": TrainingInput(
            s3_data=processing_step.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv"
        ),
        "validation": TrainingInput(
            s3_data=processing_step.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv"
        )
    }
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


## 7. Creating the Evaluation Logic (`evaluation.py`)

Once we have a best model, we need an impartial judge to grade its performance.

We write a script called `evaluation.py` to handle this. It loads the best model from the tuning step and tests it against the **Test Set**, data the model has never seen before. It calculates the **Root Mean Squared Error (RMSE)** to quantify exactly how far off the model's predictions are from reality, saving the score to a JSON file.

In [7]:
%%writefile evaluation.py
import json
import tarfile
import pandas as pd
import xgboost as xgb
import argparse
import os

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-tar", type=str)
    parser.add_argument("--test-data", type=str)
    parser.add_argument("--output-metrics", type=str)
    return parser.parse_args()

if __name__ == "__main__":
    args = parse_args()
    
    # Extract model
    with tarfile.open(args.model_tar, "r:gz") as tar:
        tar.extractall(".")
    
    # Load model
    model = xgb.Booster()
    model.load_model("xgboost-model")
    
    # Load test data
    df = pd.read_csv(args.test_data, header=None)
    y_test = df.iloc[:, 0].values
    X_test = df.iloc[:, 1:].values
    
    # Predict
    dtest = xgb.DMatrix(X_test)
    preds = model.predict(dtest)
    
    # Compute metrics
    from sklearn.metrics import mean_squared_error
    rmse = mean_squared_error(y_test, preds, squared=False)
    
    # Save metrics
    metrics = {"rmse": rmse}
    with open(os.path.join(args.output_metrics, "evaluation.json"), "w") as f:
        json.dump(metrics, f)

Overwriting evaluation.py


### 8. Configuring the Evaluation Step

This step executes the grading logic. We define a second **ProcessingStep** that spins up another `ml.t3.medium` instance.

Crucially, this step takes two inputs: the **Best Model Artifact** (from the tuning step) and the **Test Data** (from the processing step). It outputs a `PropertyFile`, which exposes the final RMSE score to the pipeline so we can make automated decisions based on the model's quality.

In [8]:
from sagemaker.processing import ScriptProcessor
from sagemaker.workflow.properties import PropertyFile

# 1. Define the PropertyFile FIRST
evaluation_json = PropertyFile(
    name="evaluation",
    output_name="metrics",
    path="evaluation.json"
)

# 2. Set up the ScriptProcessor
evaluation_processor = ScriptProcessor(
    image_uri=container,  
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.t3.medium"
)

# 3. Get the best model artifact
best_model_artifact = tuning_step.get_top_model_s3_uri(
    top_k=0,
    s3_bucket=bucket, 
    prefix=f"{prefix}/models"
)

# 4. Create the ProcessingStep and pass property_files directly into it
evaluation_step = ProcessingStep(
    name="EvaluateModel",
    processor=evaluation_processor,
    inputs=[
        ProcessingInput(source=best_model_artifact, destination="/opt/ml/processing/model"),
        ProcessingInput(
            source=processing_step.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="metrics", 
            source="/opt/ml/processing/metrics", 
            destination=f"s3://{bucket}/{prefix}/evaluation"
        )
    ],
    code="evaluation.py",
    job_arguments=[
        "--model-tar", "/opt/ml/processing/model/model.tar.gz",
        "--test-data", "/opt/ml/processing/test/test.csv",
        "--output-metrics", "/opt/ml/processing/metrics"
    ],
    property_files=[evaluation_json]
)

## 9. The Quality Gate

We don't want to register bad models. This **Condition Step** acts as a checkpoint.

It reads the RMSE score calculated in the previous step and compares it against a threshold (in this case, 0.5).
* **If the model is accurate enough (RMSE <= 0.5):** The pipeline allows it to proceed to registration.
* **If the model is inaccurate:** The pipeline stops immediately, preventing a poor model from entering our system.

In [9]:
condition = ConditionLessThanOrEqualTo(
    left=JsonGet(
        step_name=evaluation_step.name,
        property_file=evaluation_json,
        json_path="rmse"
    ),
    right=0.5  # Example threshold – adjust based on your data
)

## 10. Registering the Model

If our model passes the quality gate, we catalogue it in the **SageMaker Model Registry**.

This step creates a new version of the model in the `DiabetesModelGroup`. It stores not just the model files, but also the metadata, including the training metrics and the instance types it supports (`ml.m5.large`). This creates a centralized "Model Store" where we can track, version, and approve models for deployment.

In [10]:
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.model_step import ModelStep

# 1. Initialize a Pipeline-specific session
pipeline_session = PipelineSession()

# 2. Define the Model, passing the pipeline_session instead of the standard session
model = Model(
    image_uri=container,
    model_data=best_model_artifact,
    role=role,
    sagemaker_session=pipeline_session  # <--- FIX: This delays execution
)

# 3. Define the metrics source
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(on="/", values=[
            evaluation_step.properties.ProcessingOutputConfig.Outputs["metrics"].S3Output.S3Uri,
            "evaluation.json"
        ]),
        content_type="application/json"
    )
)

# 4. Create the Register Step
register_step = ModelStep(
    name="RegisterModel",
    step_args=model.register(
        content_types=["text/csv"],
        response_types=["text/csv"],
        inference_instances=["ml.m5.large"],
        transform_instances=["ml.m5.large"],
        model_package_group_name="DiabetesModelGroup",
        model_metrics=model_metrics,
        approval_status=model_approval_status
    )
)

# 5. Wrap it in the Condition Step (from Step 6)
condition_step = ConditionStep(
    name="CheckEvaluation",
    conditions=[condition],
    if_steps=[register_step],
    else_steps=[]
)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


## 11. Assembling the Pipeline

Here, we stitch all the individual steps—Processing, Tuning, Evaluation, and Conditional Registration—into a single, cohesive workflow object. This defines the flowchart that SageMaker will execute.

In [11]:
pipeline = Pipeline(
    name="DiabetesMLOpsPipeline",
    parameters=[
        input_data_url,
        processing_instance_count,
        training_instance_type,
        model_approval_status
    ],
    steps=[processing_step, tuning_step, evaluation_step, condition_step],
    sagemaker_session=sagemaker_session
)

## 12. Uploading the Definition

Running `pipeline.upsert` compiles our Python code into a JSON definition and sends it to the AWS SageMaker service. This registers the pipeline in your AWS account, making it ready to run.

In [12]:
pipeline.upsert(role_arn=role)

{'PipelineArn': 'arn:aws:sagemaker:ap-southeast-2:406682760260:pipeline/DiabetesMLOpsPipeline',
 'ResponseMetadata': {'RequestId': 'd24fc368-acaf-47da-85b9-f37cdff8faca',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'd24fc368-acaf-47da-85b9-f37cdff8faca',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '117',
   'date': 'Tue, 24 Feb 2026 22:42:00 GMT'},
  'RetryAttempts': 0}}

## 13. Executing the Pipeline

Finally, we hit the launch button. We start a new execution of the pipeline, passing in the specific location of our raw data. The `.wait()` command pauses the notebook, allowing us to watch the pipeline status as it provisions resources, trains models, and evaluates results in the cloud.

In [13]:
execution = pipeline.start(
    parameters=dict(
        InputDataUrl=f"s3://{bucket}/{prefix}/data/raw/diabetes.csv",
        ProcessingInstanceCount=1,
        TrainingInstanceType="ml.m5.large",
        ModelApprovalStatus="PendingManualApproval"
    )
)
execution.wait()

## 14. Verifying Success

Once the execution completes, we use these commands to inspect the results. We can list the steps to verify that the conditional logic worked as expected—confirming that our model passed the evaluation and was successfully registered.

In [14]:
execution.describe()
execution.list_steps()

[{'StepName': 'CheckEvaluation',
  'StartTime': datetime.datetime(2026, 2, 24, 22, 57, 16, 922000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 2, 24, 22, 57, 17, 86000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'Condition': {'Outcome': 'False'}},
  'AttemptCount': 1},
 {'StepName': 'EvaluateModel',
  'StartTime': datetime.datetime(2026, 2, 24, 22, 52, 13, 942000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 2, 24, 22, 57, 16, 660000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'ProcessingJob': {'Arn': 'arn:aws:sagemaker:ap-southeast-2:406682760260:processing-job/pipelines-t7xp4n9wbulm-EvaluateModel-WpgXFLwWcZ'}},
  'AttemptCount': 1},
 {'StepName': 'HyperparameterTuning',
  'StartTime': datetime.datetime(2026, 2, 24, 22, 47, 5, 277000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 2, 24, 22, 52, 13, 396000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'TuningJob': {'Arn': 'arn:aws:sagemaker:ap-so